[![Mở trong Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/trungnguyenvn/mlsysbook-vi-notebooks/blob/main/vol1-ch16-conclusion/01-muoi-ba-nguyen-ly-thanh-ma.ipynb)

# Mười ba nguyên lý thành mã: một token của Llama 2 70B đi qua cả tập sách

Chương kết gom mười ba nguyên lý định lượng của cả tập vào bảng 2. Sách nói rõ chúng không cùng một
loại: có cận toán học chính xác, có phép phân rã kỹ thuật, có mô hình khớp cục bộ, có yêu cầu chính
sách, và có nguyên tắc thiết kế kinh nghiệm. Giá trị của mỗi hàng đến từ việc áp nó trong đúng giả định
của nó.

Sổ tay này viết mỗi nguyên lý định lượng thành một hàm nhỏ, rồi áp vào cùng một ví dụ xuyên suốt: phép
tính nhẩm 1.1 của chương, phục vụ một token của Llama 2 70B trên hai H100. Nguyên lý nào ví dụ ấy không
chạm tới thì sổ tay dùng một ví dụ khác của chính chương, hoặc một thí nghiệm nhỏ đo trên máy này. Với
mỗi nguyên lý, sổ tay ghi nó thuộc phần nào của tập sách và đã được dùng ở chương nào. Chỗ nào không có
gì để tính, sổ tay nói thẳng.

**Bạn sẽ làm:**
1. dựng ví dụ xuyên suốt, rồi áp định luật sắt (nguyên lý 3) và hợp đồng với silicon (nguyên lý 4);
2. gom batch và lượng tử hoá trên biểu đồ roofline (nguyên lý 6), đếm năng lượng theo sự kiện (nguyên
   lý 7), và đặt trần Amdahl (nguyên lý 8);
3. dựng một biên Pareto đo được (nguyên lý 5), rồi đặt ngân sách độ trễ lên nó (nguyên lý 12);
4. đo ba nguyên lý triển khai trên `load_digits`: khoảng trống kiểm chứng, trôi thống kê, lệch giữa
   huấn luyện và phục vụ (nguyên lý 9 đến 11), và mô phỏng mô hình phản hồi thiên lệch (nguyên lý 13);
5. viết hai nguyên lý của phần nền tảng thành mã (nguyên lý 1 và 2);
6. lần theo một quyết định lượng tử hoá qua nhiều nguyên lý cùng lúc, rồi đổi phạm vi: từ một node
   tới cả cụm, và tới một hệ thống ghép nhiều thành phần.

In [ ]:
#@title Chuẩn bị: thư viện, hạt giống và hai hàm đối chiếu với sách { display-mode: "form" }
# Chạy ô này trước. Nó không cài thêm gì: mọi thư viện dùng ở đây đều có sẵn trên Colab.
import math, os, random, re, sys, time

import matplotlib.pyplot as plt
import numpy as np
import torch

# Số luồng CPU. 0 để thư viện tự chọn (trên Colab là 2).
NB_THREADS = int(os.environ.get("NB_THREADS", "0"))
if NB_THREADS:
    torch.set_num_threads(NB_THREADS)
    try:
        torch.set_num_interop_threads(NB_THREADS)
    except RuntimeError:  # chỉ đặt được một lần mỗi phiên
        pass
    from threadpoolctl import threadpool_limits
    threadpool_limits(NB_THREADS)

SEED = 42


def dat_hat_giong(seed=SEED):
    """Cùng hạt giống thì cùng kết quả, mỗi lần chạy lại."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


dat_hat_giong()
DATA = os.environ.get("NB_DATA", "data")  # nơi tải dữ liệu về
NB_STRICT = os.environ.get("NB_STRICT") == "1"
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 110,
                     "axes.grid": True, "grid.alpha": 0.3})

# Đọc một con số theo cách bảng số liệu của bài học viết nó (dấu chấm hay dấu phẩy).
import math
import re

_SUP = str.maketrans("⁰¹²³⁴⁵⁶⁷⁸⁹⁻⁺", "0123456789-+")
_MULT = {"tỷ": 1e9, "tỉ": 1e9, "triệu": 1e6, "nghìn": 1e3, "ngàn": 1e3,
         "K": 1e3, "M": 1e6, "B": 1e9}
_NUM = re.compile(
    r"(?<![\w.,])(-?)(\d+(?:[.,]\d+)*)"                      # sign, digits with separators
    r"(?:\s*[×x·]\s*10\s*(?:\^\{?\s*([-+]?\d+)\s*\}?|([⁻⁺]?[⁰¹²³⁴⁵⁶⁷⁸⁹]+))|[eE]([-+]?\d+)"
    r"|\^\{?\s*([-+]?\d+)\s*\}?|([⁻⁺]?[⁰¹²³⁴⁵⁶⁷⁸⁹]+))?"                  # or a bare power, 10^{-5}
    r"(\s*%|\s*(?:tỷ|tỉ|triệu|nghìn|ngàn)(?!\w)|\s?[KMB](?![A-Za-z]))?")


def norm(text):
    """The outline's markup and typography reduced to plain text, as both sides compare it."""
    text = text.replace("{,}", ",").replace("**", "").replace("`", "")
    text = re.sub(r"[   ]", " ", text)
    text = re.sub(r"−(?=\d)", "-", text)
    text = re.sub(r"[~≈]", "", text)
    return re.sub(r"\s+", " ", text).strip()


def _readings(digits):
    """(value, decimals) for each way the separators in `digits` can be read."""
    out = []
    groups_ok = lambda parts: all(len(p) == 3 for p in parts[1:])
    for thou, dec in ((",", "."), (".", ",")):
        if thou in digits and dec in digits:
            head, _, tail = digits.rpartition(dec)
            if dec in head or not tail.isdigit() or not groups_ok(head.split(thou)):
                continue
            out.append((float(head.replace(thou, "") + "." + tail), len(tail)))
    for sep in (",", "."):
        other = "." if sep == "," else ","
        if sep in digits and other not in digits:
            parts = digits.split(sep)
            if groups_ok(parts):
                out.append((float("".join(parts)), 0))                    # thousands
            if len(parts) == 2:
                out.append((float(parts[0] + "." + parts[1]), len(parts[1])))  # decimal
    if "." not in digits and "," not in digits:
        out.append((float(digits), 0))
    return out


def candidates(text):
    """Every (value, half_unit) the numbers in `text` can mean.

    half_unit is half of the last printed digit, scaled like the value, so `437,5 KB`
    accepts 437.544. Each number is offered as written and, when a multiplier or `%`
    follows it, scaled by that too, so `8,5 tỉ` cites 8.5 or 8.5e9 and `20 %` cites 20 or 0.2.
    """
    out = []
    for m in _NUM.finditer(norm(text)):
        sign, digits, exp_caret, exp_sup, exp_e, pow_caret, pow_sup, suffix = m.groups()
        exp = exp_caret or (exp_sup.translate(_SUP) if exp_sup else None) or exp_e
        power = pow_caret or (pow_sup.translate(_SUP) if pow_sup else None)
        scale10 = 10.0 ** int(exp) if exp else 1.0
        suffix = (suffix or "").strip()
        extra = 0.01 if suffix == "%" else _MULT.get(suffix)
        for value, decimals in _readings(digits):
            if power:
                # 10^{-5}, 2³²: an exact value, so it is matched to 1 part in 10⁹; a looser
                # comparison needs an explicit tol.
                v = value ** int(power)
                half = abs(v) * 1e-9
            else:
                v = value * scale10
                half = 0.5 * 10.0 ** -decimals * scale10
            v = -v if sign else v
            out.append((v, half))
            if extra:
                out.append((v * extra, half * extra))
    return out


def match(value, text):
    """The (value, half_unit) reading of `text` that `value` equals, or None."""
    for v, half in candidates(text):
        if math.isclose(value, v, rel_tol=1e-9, abs_tol=1e-300):
            return v, half
    return None


def well_anchored(text):
    """False for a citation too bare to identify one cell, like "1" or "2"."""
    t = norm(text)
    rest = re.sub(r"[\d.,\s\-+]", "", t)
    if rest:
        return True
    sig = re.sub(r"[^\d]", "", t).lstrip("0")
    return len(sig) >= 2


def _so(x):
    """Một con số để in: số nguyên có dấu phẩy ngăn nghìn, số thực giữ 6 chữ số có nghĩa."""
    if isinstance(x, (int, np.integer)) or (isinstance(x, float) and x.is_integer() and abs(x) < 1e15):
        return f"{int(x):,}"
    return f"{x:,.6g}"


def sach(gia_tri, nguon=None, trich=None, tol=None):
    """Một con số của sách, kèm đúng chữ của nó: trong bảng số liệu của bài học (nguon)
    hoặc nguyên văn trong chương (trich)."""
    chu = nguon if nguon is not None else trich
    if match(gia_tri, chu) is None:
        raise ValueError(f"{gia_tri!r} không phải con số ghi trong {chu!r}")
    return gia_tri


def theo_sach(ten, gia_tri, sach, nguon=None, trich=None, tol=None):
    """So một con số tính được với con số của sách. In ✓ khi khớp, ✗ kèm độ lệch khi không."""
    chu = nguon if nguon is not None else trich
    khop = match(sach, chu)
    if khop is None:
        raise ValueError(f"{sach!r} không phải con số ghi trong {chu!r}")
    sai_so = tol if tol is not None else khop[1]
    dung = abs(gia_tri - sach) <= sai_so
    print(f"{'✓' if dung else '✗'} {ten}: tính được {_so(gia_tri)} · sách: {chu}")
    if not dung:
        loi = f"lệch {_so(gia_tri - sach)}, quá sai số cho phép {_so(sai_so)}"
        if NB_STRICT:
            raise AssertionError(f"{ten}: {loi}")
        print(f"   {loi}. Nếu bạn vừa đổi tham số ở trên thì đây là điều được chờ đợi.")
    # None, not the bool: a cell ending in theo_sach(...) would otherwise echo True under the ✓.


def do_tren_may(ten, gia_tri, don_vi=""):
    """Một con số ĐO trên máy đang chạy: mỗi máy một khác, không phải con số của sách."""
    print(f"⏱ {ten}: {_so(gia_tri)} {don_vi} (đo trên máy này)".replace("  ", " "))


print(f"Python {sys.version.split()[0]} · PyTorch {torch.__version__} · NumPy {np.__version__}"
      f" · {torch.get_num_threads()} luồng CPU")

In [ ]:
import warnings
from scipy import stats
from sklearn.datasets import load_digits
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline

warnings.filterwarnings("ignore", category=UserWarning)
MAU = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#8e5cc4"]  # thứ tự màu cố định
XAM = "0.45"
print("sẵn sàng")

## 0. Ví dụ xuyên suốt: phép tính nhẩm 1.1

Phép tính nhẩm 1.1 phục vụ một token của Llama 2 70B ở batch 1, với trọng số FP16 chia đều cho hai
H100. Bảng tra phần cứng của sách cho băng thông gộp của hai GPU và thông lượng đỉnh FP16 gộp. Phép tính
cố ý bỏ qua lưu lượng của bộ nhớ đệm KV, giá trị kích hoạt, phép tính attention trên bộ nhớ đệm, truyền
thông giữa hai GPU và phụ trội điều phối. Mọi con số dưới đây vì thế là cận dưới lý tưởng.

In [ ]:
THAM_SO = sach(70e9, nguon="70 tỷ · 2 byte · 140 GB · 140 GFLOP")
BYTE_FP16 = sach(2, nguon="70 tỷ · 2 byte · 140 GB · 140 GFLOP")
BW = sach(6.70, nguon="2 H100 · 6.70 TB/s · ≈1,978 TFLOP/s", tol=0.005) * 1e12      # byte/s, gộp
R_PEAK = sach(1978, nguon="2 H100 · 6.70 TB/s · ≈1,978 TFLOP/s", tol=0.5) * 1e12    # FLOP/s FP16, gộp
SO_GPU = sach(2, nguon="2 H100 · 6.70 TB/s · ≈1,978 TFLOP/s", tol=0)
D_VOL = THAM_SO * BYTE_FP16   # byte trọng số phải đọc cho một token
O = 2 * THAM_SO               # mỗi tham số một phép nhân và một phép cộng
theo_sach("khối lượng trọng số di chuyển (GB)", D_VOL / 1e9, sach=140, nguon="70 tỷ · 2 byte · 140 GB · 140 GFLOP")
theo_sach("số phép tính mỗi token (GFLOP)", O / 1e9, sach=140, nguon="70 tỷ · 2 byte · 140 GB · 140 GFLOP")

## 1. Phần II, xây dựng: định luật sắt và hợp đồng với silicon

### Nguyên lý 3: định luật sắt

Định luật sắt (iron law) mở đầu cả tập sách: chương 1 giới thiệu nó, chương 2 dùng nó để phân tích tải
công việc, và chương 8 xác nhận rằng thời gian huấn luyện giảm nhiều nhất khi tối ưu đúng số hạng chi
phối. Thời gian của một chặng gồm ba số hạng: di chuyển dữ liệu $D_{vol}/BW$, tính toán
$O/(R_{peak}\,\eta_{hw})$ và độ trễ cố định $L_{lat}$. Bảng 2 thêm một chú ý: tuỳ mức chồng lấn, tổng
thật nằm giữa số hạng lớn nhất và tổng cả ba.

### Dự đoán

Trong hai cận dưới của một token, truyền bộ nhớ và tính toán ở thông lượng đỉnh, cận nào lớn hơn, và lớn
hơn khoảng bao nhiêu lần?

In [ ]:
def dinh_luat_sat(d_vol, bw, o, r_peak, eta=1.0, l_lat=0.0):
    """Ba số hạng của định luật sắt (giây), và hai đầu của khoảng chồng lấn."""
    t_mem, t_comp = d_vol / bw, o / (r_peak * eta)
    return dict(t_mem=t_mem, t_comp=t_comp, chong_het=max(t_mem, t_comp) + l_lat,
                noi_tiep=t_mem + t_comp + l_lat)


tok = dinh_luat_sat(D_VOL, BW, O, R_PEAK)
theo_sach("cận dưới truyền bộ nhớ (ms)", 1e3 * tok["t_mem"], sach=20.9,
          nguon="≈20.9 ms · ≈0.07 ms · 295.2× · ≈1 FLOP mỗi byte", tol=0.05)
theo_sach("cận dưới tính toán đỉnh (ms)", 1e3 * tok["t_comp"], sach=0.07,
          nguon="≈20.9 ms · ≈0.07 ms · 295.2× · ≈1 FLOP mỗi byte", tol=0.005)
theo_sach("tỉ số giữa hai cận dưới", tok["t_mem"] / tok["t_comp"], sach=295.2,
          nguon="≈20.9 ms · ≈0.07 ms · 295.2× · ≈1 FLOP mỗi byte", tol=0.05)
print(f"chồng hết: {1e3 * tok['chong_het']:.2f} ms · nối tiếp: {1e3 * tok['noi_tiep']:.2f} ms · "
      f"trần một chuỗi: {1 / tok['t_mem']:.1f} token mỗi giây")

Ở đây khoảng chồng lấn hẹp tới mức không đáng kể, vì một số hạng lấn át số hạng kia. Định luật sắt
chỉ ra nên kéo đòn bẩy nào: chỉ số hạng bộ nhớ.

### Nguyên lý 4: hợp đồng với silicon

Hợp đồng với silicon so mật độ tính toán (arithmetic intensity) của mô hình, tức số phép tính trên mỗi
byte di chuyển, với điểm gãy (ridge point) của phần cứng, $I_{ridge} = R_{peak}/BW$. Thấp hơn điểm gãy
là nghẽn ở băng thông, cao hơn là nghẽn ở tính toán. Chương 2 dựng phép cân bằng ấy, chương 5 cho thấy
phép nhân ma trận quyết định mật độ tính toán, và chương 7 cho thấy framework ràng buộc những con đường
triển khai còn mở.

In [ ]:
def hop_dong_silicon(o, d_vol, r_peak, bw):
    """(mật độ tính toán của mô hình, điểm gãy của phần cứng, chế độ nghẽn)."""
    i_mo_hinh, i_gay = o / d_vol, r_peak / bw
    return i_mo_hinh, i_gay, "nghẽn ở băng thông" if i_mo_hinh < i_gay else "nghẽn ở tính toán"


I_MO_HINH, I_GAY, CHE_DO = hop_dong_silicon(O, D_VOL, R_PEAK, BW)
theo_sach("mật độ tính toán ở batch 1 (FLOP mỗi byte)", I_MO_HINH, sach=1,
          nguon="≈20.9 ms · ≈0.07 ms · 295.2× · ≈1 FLOP mỗi byte", tol=0.05)
print(f"điểm gãy của hai H100: {I_GAY:.1f} FLOP mỗi byte → {CHE_DO}")
assert math.isclose(tok["t_mem"] / tok["t_comp"], I_GAY / I_MO_HINH)
print("tỉ số hai cận dưới của định luật sắt = điểm gãy ÷ mật độ tính toán: hai nguyên lý là một phép so")

Tỉ số 295.2 lần của định luật sắt và vị trí của mô hình so với điểm gãy là cùng một con số nhìn từ hai
phía. Đó là lý do bảng 2 đặt hai nguyên lý cạnh nhau trong phần xây dựng.

## 2. Phần III, tối ưu: mật độ tính toán, năng lượng, Amdahl

### Nguyên lý 6: định luật mật độ tính toán

Thông lượng đạt được không vượt quá $\min(R_{peak},\ I \times BW)$. Thêm tính toán không nâng được một
trần băng thông. Chương 11 dựng biểu đồ roofline từ đúng bất đẳng thức này. Phép tính nhẩm 1.1 kết thúc
bằng hai đòn bẩy: gom batch tăng mức tái dùng, còn lượng tử hoá giảm lưu lượng trọng số. Mô hình dưới
đây giữ đúng các giả định đã bỏ qua của sách: ở batch $B$, một bước giải mã đọc trọng số một lần cho
$B$ token, nên mật độ tính toán tăng $B$ lần. Bảng 1 của chương ghi FP16 tốn gấp đôi dung lượng mỗi giá
trị so với INT8.

### Dự đoán

Gom 8 chuỗi vào một batch thì thời gian cho mỗi token giảm bao nhiêu lần? Từ batch nào trở đi, gom thêm
không còn làm mỗi token rẻ hơn?

In [ ]:
SO_BIT = {"FP32": 32, "FP16": 16, "INT8": 8}   # độ rộng của mỗi kiểu số
assert BYTE_FP16 == SO_BIT["FP16"] / 8
BYTE_INT8 = SO_BIT["INT8"] / 8
theo_sach("FP16 so với INT8, dung lượng mỗi giá trị", BYTE_FP16 / BYTE_INT8, sach=2, nguon="4× · 2×")
print(f"byte mỗi giá trị: FP32 {SO_BIT['FP32'] / 8:g} · FP16 {BYTE_FP16:g} · INT8 {BYTE_INT8:g}")


def thong_luong_dat(i, r_peak=R_PEAK, bw=BW):
    """Định luật mật độ tính toán: R_attain ≤ min(R_peak, I × BW)."""
    return np.minimum(r_peak, np.asarray(i, dtype=float) * bw)


def mot_buoc_giai_ma(batch, byte_moi_tham_so=BYTE_FP16):
    """Thời gian một bước giải mã cho `batch` chuỗi, cận dưới lý tưởng, chồng hết hai số hạng."""
    t = dinh_luat_sat(THAM_SO * byte_moi_tham_so, BW, batch * O, R_PEAK)
    return max(t["t_mem"], t["t_comp"])


CO_BATCH = np.unique(np.r_[np.geomspace(1, 4096, 49).round(), 8, 295, 296]).astype(int)
moi_token = {ten: np.array([mot_buoc_giai_ma(b, by) / b for b in CO_BATCH])
             for ten, by in (("FP16", BYTE_FP16), ("INT8", BYTE_INT8))}
moi_buoc = {ten: np.array([mot_buoc_giai_ma(b, by) for b in CO_BATCH])
            for ten, by in (("FP16", BYTE_FP16), ("INT8", BYTE_INT8))}
print(f"thông lượng đạt được ở batch 1: {thong_luong_dat(I_MO_HINH) / 1e12:.2f} TFLOP/s, "
      f"tức {thong_luong_dat(I_MO_HINH) / R_PEAK:.2%} thông lượng đỉnh")
for b in (1, 8, 64):
    print(f"batch {b:>3}, FP16: {1e3 * mot_buoc_giai_ma(b) / b:7.3f} ms mỗi token · "
          f"{1e3 * mot_buoc_giai_ma(b):6.2f} ms mỗi bước")
print(f"INT8 ở batch 1: {1e3 * mot_buoc_giai_ma(1, BYTE_INT8):.2f} ms mỗi token")
b_gay = int(np.ceil(I_GAY))
print(f"FP16 chạm điểm gãy ở batch {b_gay}; INT8 ở batch {int(np.ceil(I_GAY * BYTE_INT8 / BYTE_FP16))}")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.4))
i_quet = np.geomspace(0.3, 3000, 200)
ax1.plot(i_quet, thong_luong_dat(i_quet) / 1e12, color="0.25", lw=2, label="trần: min(R_peak, I × BW)")
for k, (b, by, ten) in enumerate(((1, BYTE_FP16, "FP16, batch 1"), (8, BYTE_FP16, "FP16, batch 8"),
                                  (64, BYTE_FP16, "FP16, batch 64"), (1, BYTE_INT8, "INT8, batch 1"))):
    i = b * O / (THAM_SO * by)
    ax1.scatter([i], [thong_luong_dat(i) / 1e12], s=40, color=MAU[k], zorder=5, label=ten)
ax1.axvline(I_GAY, color=XAM, ls="--", lw=1)
ax1.text(I_GAY * 1.1, 3, f"điểm gãy\n{I_GAY:.0f} FLOP/byte", fontsize=8, color=XAM)
ax1.set_xscale("log")
ax1.set_yscale("log")
ax1.set_xlabel("mật độ tính toán (FLOP mỗi byte, thang log)")
ax1.set_ylabel("thông lượng đạt được (TFLOP/s, thang log)")
ax1.set_title("Biểu đồ roofline của hai H100", fontsize=10)
ax1.legend(loc="upper left", fontsize=8)

for k, ten in enumerate(moi_token):
    ax2.plot(CO_BATCH, 1e3 * moi_token[ten], color=MAU[k], lw=2, label=f"{ten}: mỗi token")
    ax2.plot(CO_BATCH, 1e3 * moi_buoc[ten], color=MAU[k], lw=1.2, ls="--", label=f"{ten}: mỗi bước")
ax2.set_xscale("log", base=2)
ax2.set_yscale("log")
ax2.set_xlabel("số chuỗi trong một batch (thang log)")
ax2.set_ylabel("thời gian, cận dưới lý tưởng (ms, thang log)")
ax2.set_title("Gom batch: mỗi token rẻ đi cho tới điểm gãy", fontsize=10)
ax2.legend(loc="lower left", fontsize=8)
plt.tight_layout()
plt.show()

Bên trái, ở batch 1 mô hình nằm sát chân dốc của trần băng thông, cách điểm gãy hơn hai bậc. Gom batch
đẩy điểm ấy sang phải dọc theo dốc. Bên phải, thời gian của mỗi token giảm theo đúng số chuỗi trong
batch cho tới điểm gãy, rồi đứng yên, vì từ đó số hạng tính toán chi phối. Còn thời gian của mỗi bước,
là độ trễ mà mọi chuỗi trong batch cùng chịu, thì phẳng rồi tăng. INT8 dời cả hai đường xuống một nửa
trong vùng nghẽn ở băng thông. Các giả định mà sách bỏ qua, nhất là bộ nhớ đệm KV lớn dần theo độ dài
ngữ cảnh, sẽ kéo điểm gãy thật về phía trái.

### Nguyên lý 7: bất biến năng lượng di chuyển

Tổng năng lượng là tổng theo loại sự kiện của số lần nhân chi phí mỗi lần, $E_{total} = \sum_j N_j E_j$.
Bất biến này được hình thức hoá ở chương 4 và định lượng ở chương 12. Theo các hằng số tham chiếu của
sách, một lần truy cập DRAM tốn khoảng 173 tới 582 lần năng lượng của một phép tính FP32 hay FP16. Sách
nhấn mạnh: loại nào chi phối tổng năng lượng còn tuỳ mỗi loại xảy ra bao nhiêu lần. Sách không nói một
lần truy cập là bao nhiêu byte, nên ô dưới giả định một lần truy cập đọc một giá trị FP16.

In [ ]:
TI_LE_DRAM = (sach(173, nguon="173–582×"), sach(582, nguon="173–582×"))  # năng lượng DRAM ÷ năng lượng một phép tính


def nang_luong(so_lan, chi_phi):
    """Bất biến năng lượng di chuyển: E_total = Σ N_j · E_j."""
    return sum(n * e for n, e in zip(so_lan, chi_phi))


for r in TI_LE_DRAM:
    for b in (1, 8, 64, 512):
        n_truy_cap, n_phep_tinh = THAM_SO / b, O   # cho mỗi token: trọng số chia cho cả batch
        e_dram = nang_luong([n_truy_cap], [r])
        e_tong = nang_luong([n_truy_cap, n_phep_tinh], [r, 1.0])
        print(f"tỉ lệ {r:3.0f} lần, batch {b:>3}: DRAM chiếm {e_dram / e_tong:6.1%} năng lượng mỗi token "
              f"(gấp {e_dram / (e_tong - e_dram):6.1f} lần phần tính toán)")
print(f"DRAM thôi chi phối khi batch vượt {TI_LE_DRAM[0] / 2:.1f} tới {TI_LE_DRAM[1] / 2:.0f} chuỗi (giả định một giá trị FP16 mỗi lần truy cập)")

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 4))
b_quet = np.geomspace(1, 4096, 100)
for k, r in enumerate(TI_LE_DRAM):
    phan = (THAM_SO / b_quet * r) / (THAM_SO / b_quet * r + O)
    ax.plot(b_quet, 100 * phan, color=MAU[k], lw=2, label=f"DRAM tốn gấp {r:g} lần một phép tính")
ax.axhline(50, color=XAM, ls="--", lw=1)
ax.text(1.2, 52, "một nửa", fontsize=8, color=XAM)
ax.set_xscale("log", base=2)
ax.set_ylim(0, 105)
ax.set_xlabel("số chuỗi trong một batch (thang log)")
ax.set_ylabel("phần năng lượng mỗi token cho DRAM (%)")
ax.set_title("Cùng chi phí mỗi sự kiện, số lần sự kiện quyết định bên nào chi phối", fontsize=10)
ax.legend(loc="lower left", fontsize=8)
plt.tight_layout()
plt.show()

Ở batch 1, truy cập DRAM nuốt gần hết năng lượng của một token, dù với cận thấp nhất của tỉ lệ. Gom batch
không đổi chi phí của một lần truy cập, nó đổi số lần truy cập ứng với mỗi token. Đó chính là ý của bảng
2: "tổng năng lượng phụ thuộc số lần và chi phí của mỗi sự kiện".

### Nguyên lý 8: định luật Amdahl

Phần không được tăng tốc đặt trần cho mọi mức tăng tốc: $1/((1-f) + f/S)$, với $f$ là phần thời gian
được tăng tốc $S$ lần. Chương 4 áp nó cho quy trình dữ liệu, chương 11 cho bộ tăng tốc, và chương 12
biến nó thành phương pháp đo xem thời gian thật sự đi đâu. Mục ngộ nhận của chương kết đưa một ví dụ:
tăng tốc phần suy luận trên bộ tăng tốc 10 lần, khi tiền xử lý trên CPU chiếm 90 phần trăm thời gian.

In [ ]:
def amdahl(f, s):
    """Mức tăng tốc cả hệ thống khi phần f được tăng tốc s lần."""
    return 1.0 / ((1.0 - f) + f / s)


S_NHANH = sach(10, nguon="10× · 90 phần trăm · 1.1×")
PHAN_CPU = sach(90, nguon="10× · 90 phần trăm · 1.1×") / 100
theo_sach("tăng tốc cả hệ thống", amdahl(1 - PHAN_CPU, S_NHANH), sach=1.1, nguon="10× · 90 phần trăm · 1.1×")
f_tinh_toan = tok["t_comp"] / tok["noi_tiep"]
print(f"ví dụ xuyên suốt: số hạng tính toán chiếm {f_tinh_toan:.2%} thời gian nối tiếp; "
      f"tăng tốc nó vô hạn lần chỉ được {amdahl(f_tinh_toan, 1e12):.4f} lần")
print(f"còn giảm một nửa số hạng bộ nhớ (INT8) được {amdahl(1 - f_tinh_toan, 2):.3f} lần")

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 4))
s_quet = np.geomspace(1, 1000, 100)
for k, f in enumerate((0.1, 0.5, 0.9, 0.99)):
    ax.plot(s_quet, amdahl(f, s_quet), color=MAU[k], lw=2, label=f"phần được tăng tốc {f:.0%}")
    ax.axhline(1 / (1 - f), color=MAU[k], lw=0.8, ls=":")
ax.scatter([S_NHANH], [amdahl(1 - PHAN_CPU, S_NHANH)], color="black", zorder=5)
ax.annotate("ví dụ của mục ngộ nhận", (S_NHANH, amdahl(1 - PHAN_CPU, S_NHANH)), xytext=(30, 1.4),
            textcoords="data", fontsize=8, va="center", arrowprops=dict(arrowstyle="-", color=XAM, lw=0.8))
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("mức tăng tốc của phần được tối ưu (lần, thang log)")
ax.set_ylabel("tăng tốc cả hệ thống (lần, thang log)")
ax.set_title("Định luật Amdahl: đường chấm là trần 1 / (1 − f)", fontsize=10)
ax.legend(loc="upper left", fontsize=8)
plt.tight_layout()
plt.show()

Hai dòng in ra là cùng một bài học, nhìn từ ví dụ xuyên suốt. Tối ưu kernel tính toán của một token
gần như không đổi được gì, còn giảm một nửa số hạng bộ nhớ gần như giảm một nửa thời gian. Định luật
sắt chỉ ra số hạng chi phối, còn Amdahl cho biết tối ưu số hạng khác thì trần nằm ở đâu.

### Nguyên lý 5: biên Pareto

Một cấu hình nằm trên biên Pareto khi không cấu hình nào khác tốt bằng nó ở mọi chỉ số và tốt hơn ở ít
nhất một chỉ số, sau khi hướng của các chỉ số đã được chuẩn hoá. Chương 10 đi thẳng trên biên này khi nén
mô hình, chương 12 đo nó, và chương 15 dựng nó giữa độ chính xác và công bằng. Ô dưới viết định nghĩa
thành hàm, rồi dựng một biên đo được: mười một bộ phân loại chữ số trên `load_digits`, đo độ chính xác
và đếm tham số.

In [ ]:
def troi_hon(a, b):
    """a trội hơn b: không kém ở chỉ số nào, hơn ở ít nhất một (mọi chỉ số đều 'lớn hơn là tốt')."""
    return all(x >= y for x, y in zip(a, b)) and any(x > y for x, y in zip(a, b))


def bien_pareto(diem):
    """Chỉ số của các điểm không bị điểm nào khác trội hơn."""
    return [i for i, a in enumerate(diem) if not any(troi_hon(b, a) for j, b in enumerate(diem) if j != i)]


X_so, y_so = load_digits(return_X_y=True)
X_so = X_so / 16.0
thu_tu = np.random.default_rng(SEED).permutation(len(X_so))
hl, kt = thu_tu[:1000], thu_tu[1000:]
cau_hinh = []
t0 = time.perf_counter()
for k in (2, 4, 8, 16, 32):
    m = make_pipeline(PCA(k, random_state=SEED), LogisticRegression(max_iter=3000)).fit(X_so[hl], y_so[hl])
    cau_hinh.append((f"PCA {k} + hồi quy logistic", 64 * k + 10 * k + 10, (m.predict(X_so[kt]) == y_so[kt]).mean()))
m = LogisticRegression(max_iter=3000).fit(X_so[hl], y_so[hl])
cau_hinh.append(("hồi quy logistic", 64 * 10 + 10, (m.predict(X_so[kt]) == y_so[kt]).mean()))
for h in (4, 8, 16, 32, 64):
    m = MLPClassifier((h,), max_iter=800, random_state=SEED).fit(X_so[hl], y_so[hl])
    cau_hinh.append((f"MLP {h} nơ-ron ẩn", 64 * h + h + 10 * h + 10, (m.predict(X_so[kt]) == y_so[kt]).mean()))
do_tren_may("thời gian huấn luyện mười một mô hình", time.perf_counter() - t0, "s")
diem = [(acc, -so_tham_so) for _, so_tham_so, acc in cau_hinh]  # tham số: ít hơn là tốt, nên đổi dấu
tren_bien = bien_pareto(diem)
for i, (ten, so_tham_so, acc) in enumerate(cau_hinh):
    do_tren_may(f"{ten:>26}, {so_tham_so:>5,} tham số{', trên biên' if i in tren_bien else ''}: độ chính xác",
                100 * acc, "%")

### Nguyên lý 12: ngân sách độ trễ, và một ràng buộc đặt lên biên

Biên Pareto không chọn hộ một điểm. Một yêu cầu sản phẩm mới chọn: ngân sách độ trễ, $T_q \le L_{budget}$,
ràng buộc việc phục vụ ở phân vị $q$ mà sản phẩm chọn. Chương 13 dựng cả chương phục vụ quanh nó. Mục ngộ
nhận của chương kết đưa hai điểm: dưới mục tiêu độ trễ đuôi 100 ms, mô hình chính xác 95 phần trăm ở
500 ms không khả thi, còn mô hình 93 phần trăm ở 50 ms vẫn là ứng viên. Không có yêu cầu ấy, không điểm
nào tự động tốt hơn điểm kia.

In [ ]:
def kha_thi(diem_do_tre, ngan_sach):
    """Nguyên lý 12: giữ những cấu hình có độ trễ ở phân vị đã chọn không vượt ngân sách."""
    return [ten for ten, t_q in diem_do_tre.items() if t_q <= ngan_sach]


HAI_DIEM = {  # (độ chính xác %, độ trễ đuôi ms)
    "95 phần trăm": (sach(95, nguon="100 ms · 95 phần trăm ở 500 ms · 93 phần trăm ở 50 ms"),
                     sach(500, nguon="100 ms · 95 phần trăm ở 500 ms · 93 phần trăm ở 50 ms")),
    "93 phần trăm": (sach(93, nguon="100 ms · 95 phần trăm ở 500 ms · 93 phần trăm ở 50 ms"),
                     sach(50, nguon="100 ms · 95 phần trăm ở 500 ms · 93 phần trăm ở 50 ms")),
}
SLO_DUOI = sach(100, nguon="100 ms · 95 phần trăm ở 500 ms · 93 phần trăm ở 50 ms")
a, b = HAI_DIEM["95 phần trăm"], HAI_DIEM["93 phần trăm"]
print("điểm nào trội hơn điểm nào:", "không điểm nào" if not (troi_hon((a[0], -a[1]), (b[0], -b[1])) or
                                                             troi_hon((b[0], -b[1]), (a[0], -a[1]))) else "có")
print(f"khả thi dưới {SLO_DUOI:g} ms:", kha_thi({k: v[1] for k, v in HAI_DIEM.items()}, SLO_DUOI))

NGAN_SACH_THAM_SO = 1500  # giả định: ngân sách bộ nhớ của một thiết bị nhỏ, tính bằng tham số
vua = [i for i in tren_bien if cau_hinh[i][1] <= NGAN_SACH_THAM_SO]
chon = max(vua, key=lambda i: cau_hinh[i][2])
print(f"trên biên đo được, dưới ngân sách {NGAN_SACH_THAM_SO:,} tham số (giả định): {cau_hinh[chon][0]}")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.3), gridspec_kw=dict(width_ratios=[1.4, 1]))
so_ts = np.array([c[1] for c in cau_hinh])
dcx = np.array([100 * c[2] for c in cau_hinh])
la_mlp = np.array([c[0].startswith("MLP") for c in cau_hinh])
ax1.scatter(so_ts[~la_mlp], dcx[~la_mlp], color=MAU[0], s=30, label="PCA + hồi quy logistic")
ax1.scatter(so_ts[la_mlp], dcx[la_mlp], color=MAU[1], s=30, marker="s", label="MLP một tầng ẩn")
thu_tu_bien = sorted(tren_bien, key=lambda i: so_ts[i])
ax1.step(so_ts[thu_tu_bien], dcx[thu_tu_bien], where="post", color="0.3", lw=1.2, label="biên Pareto")
ax1.axvline(NGAN_SACH_THAM_SO, color=XAM, ls="--", lw=1)
ax1.text(NGAN_SACH_THAM_SO * 1.06, 60, "ngân sách\n(giả định)", fontsize=8, color=XAM)
ax1.scatter([so_ts[chon]], [dcx[chon]], s=140, facecolors="none", edgecolors="black", zorder=6)
ax1.set_xscale("log")
ax1.set_xlabel("số tham số (thang log)")
ax1.set_ylabel("độ chính xác trên ảnh kiểm tra (%)")
ax1.set_title("Biên Pareto đo trên load_digits (đo trên máy này)", fontsize=10)
ax1.legend(loc="upper left", fontsize=8)

for k, (ten, (acc, ms)) in enumerate(HAI_DIEM.items()):
    ax2.scatter([ms], [acc], s=60, color=MAU[2 + k], zorder=5)
    ax2.annotate(f"{ten} ở {ms:g} ms", (ms, acc), xytext=(-10, -14) if k == 0 else (8, 8),
                 textcoords="offset points", fontsize=8, ha="right" if k == 0 else "left")
ax2.axvspan(SLO_DUOI, 1000, color="0.9")
ax2.axvline(SLO_DUOI, color=XAM, ls="--", lw=1)
ax2.text(SLO_DUOI * 1.08, 96.3, "vượt mục tiêu\nđộ trễ đuôi", fontsize=8, color=XAM, va="top")
ax2.set_xscale("log")
ax2.set_xlim(20, 1000)
ax2.set_ylim(92, 96.5)
ax2.set_xlabel("độ trễ đuôi (ms, thang log)")
ax2.set_ylabel("độ chính xác (%)")
ax2.set_title("Hai điểm của mục ngộ nhận, mục tiêu 100 ms", fontsize=10)
plt.tight_layout()
plt.show()

Bên trái, mỗi mô hình nằm dưới biên đều có một mô hình khác vừa nhỏ hơn vừa chính xác hơn. Hàm
`bien_pareto` chỉ loại những điểm ấy, còn giữ lại cả một dải lựa chọn mà không điểm nào hơn hẳn điểm nào.
Ngân sách cắt dải ấy thành hai phần, và lời giải là điểm tốt nhất của phần khả thi chứ không phải điểm chính
xác nhất.
Bên phải là cùng logic với hai điểm của sách: đổi ngân sách thì đổi luôn mô hình được chọn.

### Nguyên lý 12, tiếp: phân vị nào

Chương kết dùng một kịch bản minh hoạ để nói vì sao phải chọn phân vị: độ trễ trung bình 50 ms, phân vị
99 là 2,000 ms. Ô dưới tính tỉ số, rồi dựng một phân phối giả định có đúng dáng ấy (đa số yêu cầu
nhanh, một phần nhỏ rất chậm, như khi thu gom rác hay sinh một câu trả lời dài) để xem một ngân sách đặt ở
các phân vị khác nhau nói gì.

In [ ]:
TRUNG_BINH = sach(50, nguon="50 ms · 2,000 ms · 40×")
P99 = sach(2000, nguon="50 ms · 2,000 ms · 40×")
theo_sach("phân vị 99 so với trung bình", P99 / TRUNG_BINH, sach=40, nguon="50 ms · 2,000 ms · 40×")
r_tre = np.random.default_rng(SEED)
N_YEU_CAU = 200_000
cham = r_tre.random(N_YEU_CAU) < 0.02                               # giả định: 2 % yêu cầu đi đường chậm
do_tre = np.where(cham, r_tre.lognormal(np.log(2000), 0.3, N_YEU_CAU), r_tre.lognormal(np.log(7.7), 0.4, N_YEU_CAU))
print(f"phân phối giả định: trung bình {do_tre.mean():.0f} ms · phân vị 50 {np.percentile(do_tre, 50):.1f} ms · "
      f"phân vị 99 {np.percentile(do_tre, 99):.0f} ms")
print(f"yêu cầu nằm trong ±20 % quanh trung bình: {np.mean(np.abs(do_tre - do_tre.mean()) < 0.2 * do_tre.mean()):.2%}")
NGAN_SACH_MS = 300  # giả định
for q in (50, 90, 95, 99, 99.9):
    t_q = np.percentile(do_tre, q)
    print(f"phân vị {q:>4}: {t_q:7.1f} ms → {'đạt' if t_q <= NGAN_SACH_MS else 'vượt'} ngân sách {NGAN_SACH_MS} ms")

Số trung bình 50 ms gần như không mô tả yêu cầu nào: phần lớn nhanh hơn nhiều, một phần nhỏ chậm hơn
nhiều. Cùng một hệ thống đạt ngân sách ở phân vị 95 và vượt ở phân vị 99, nên câu "hệ thống có đạt mục
tiêu không" chỉ có nghĩa khi yêu cầu sản phẩm đã chọn $q$.

## 3. Phần IV, triển khai: đo trên `load_digits`

Năm nguyên lý triển khai nhắm vào những thất bại mà kiểm thử trong phòng thí nghiệm không loại trừ được.
Ba nguyên lý đầu đo được bằng một mô hình nhỏ, và chương 14 vận hành cả ba.

### Nguyên lý 9: khoảng trống kiểm chứng

Kiểm chứng thống kê chỉ đúng trên một tổng thể triển khai đã nêu, với một khoảng cách, một dung sai, một
mức lỗi mục tiêu và một thủ tục tin cậy cho mẫu hữu hạn. Hàm đầu tiên trả lời câu hỏi đơn giản nhất:
nếu không thấy lỗi nào trong $n$ mẫu, cần $n$ bao nhiêu để nói tỉ lệ lỗi dưới $\varepsilon$ với độ tin cậy
$1 - \delta$? Các mức $\varepsilon$ và $\delta$ là giả định của sổ tay.

In [ ]:
def mau_can_khi_khong_loi(eps, delta):
    """Số mẫu không lỗi tối thiểu để kết luận tỉ lệ lỗi < eps với độ tin cậy 1 − delta."""
    return math.ceil(math.log(delta) / math.log(1 - eps))


def can_duoi_do_chinh_xac(dung, n, delta=0.05):
    """Cận dưới một phía Clopper-Pearson của độ chính xác."""
    return 0.0 if dung == 0 else float(stats.beta.ppf(delta, dung, n - dung + 1))


for eps in (0.01, 0.001, 0.0001):
    print(f"lỗi dưới {eps:.2%} ở độ tin cậy 95 %: cần {mau_can_khi_khong_loi(eps, 0.05):,} mẫu liên tiếp không lỗi")
mo_hinh = LogisticRegression(max_iter=3000).fit(X_so[hl], y_so[hl])
dung = int((mo_hinh.predict(X_so[kt]) == y_so[kt]).sum())
can = can_duoi_do_chinh_xac(dung, len(kt))
do_tren_may("độ chính xác trên tổng thể đã kiểm chứng", 100 * dung / len(kt), "%")
do_tren_may("cận dưới 95 % của nó", 100 * can, "%")
r_nhieu = np.random.default_rng(SEED + 1)
X_nhieu = np.clip(X_so[kt] + r_nhieu.normal(0, 0.25, X_so[kt].shape), 0, 1)   # giả định: cảm biến nhiễu hơn
acc_nhieu = (mo_hinh.predict(X_nhieu) == y_so[kt]).mean()
do_tren_may("độ chính xác trên tổng thể có nhiễu", 100 * acc_nhieu, "%")
print("cận dưới đã kiểm chứng còn đúng trên tổng thể mới:", "có" if acc_nhieu >= can else "không")

Cận dưới là một phát biểu về tổng thể đã lấy mẫu, không phải về mọi đầu vào tương lai. Khi tổng thể đổi,
phát biểu ấy không tự mang sang. Đó là lý do sách gọi đây là một khoảng trống: kiểm thử ước lượng hành vi
chứ không chứng minh hệ thống đúng.

### Nguyên lý 10: chẩn đoán trôi thống kê

Bảng 2 viết nguyên lý này là một phép khớp cục bộ, $\text{Accuracy}(t) \approx \text{Accuracy}_0 -
\lambda\,\mathcal{D}(P_t \Vert P_0)$, và ghi ngay bên cạnh: trôi không nhất thiết làm chất lượng giảm.
Ô dưới đo một chỉ số trôi duy nhất, phân kỳ Jensen-Shannon trên thành phần chính thứ nhất của ảnh, cho
hai kiểu dịch chuyển giả định với cường độ tăng dần. Kiểu thứ nhất thêm nhiễu vào ảnh. Kiểu thứ hai chỉ
đổi tỉ lệ các lớp, nghiêng về ba chữ số mà mô hình nhận đúng nhiều nhất.

### Dự đoán

Hai kiểu dịch chuyển cùng làm chỉ số trôi tăng. Độ chính xác có cùng giảm không?

In [ ]:
pca = PCA(2, random_state=SEED).fit(X_so[hl])
goc = pca.transform(X_so[kt])[:, 0]
can_bin = np.linspace(goc.min() - 1, goc.max() + 1, 21)


def phan_ky_js(a, b, bins=can_bin):
    """Phân kỳ Jensen-Shannon giữa hai histogram (nat), với làm trơn cộng 0.5."""
    pa = np.histogram(a, bins)[0] + 0.5
    pb = np.histogram(b, bins)[0] + 0.5
    pa, pb = pa / pa.sum(), pb / pb.sum()
    m = (pa + pb) / 2
    return 0.5 * np.sum(pa * np.log(pa / m)) + 0.5 * np.sum(pb * np.log(pb / m))


du_doan_goc = mo_hinh.predict(X_so[kt])
theo_lop = np.array([(du_doan_goc[y_so[kt] == c] == c).mean() for c in range(10)])
lop_de = np.argsort(theo_lop)[-3:]
r_troi = np.random.default_rng(SEED + 2)
CUONG_DO = np.linspace(0, 1, 6)
troi = {"thêm nhiễu": [], "đổi tỉ lệ lớp": []}
for s in CUONG_DO:
    Xn = np.clip(X_so[kt] + r_troi.normal(0, 0.5 * s, X_so[kt].shape), 0, 1)
    troi["thêm nhiễu"].append((phan_ky_js(goc, pca.transform(Xn)[:, 0]), (mo_hinh.predict(Xn) == y_so[kt]).mean()))
    w = np.where(np.isin(y_so[kt], lop_de), 1 + 6 * s, 1.0)
    chon_mau = r_troi.choice(len(kt), 3000, p=w / w.sum())
    Xp, yp = X_so[kt][chon_mau], y_so[kt][chon_mau]
    troi["đổi tỉ lệ lớp"].append((phan_ky_js(goc, pca.transform(Xp)[:, 0]), (mo_hinh.predict(Xp) == yp).mean()))
for ten, ds in troi.items():
    for s, (d, acc) in zip(CUONG_DO, ds):
        do_tren_may(f"{ten}, cường độ {s:.1f}: phân kỳ {d:.4f}, độ chính xác", 100 * acc, "%")
d_nhieu, a_nhieu = np.array(troi["thêm nhiễu"]).T
lam, acc0 = np.polyfit(d_nhieu[:3], a_nhieu[:3], 1)   # khớp cục bộ trên ba điểm trôi nhẹ của kiểu nhiễu
print(f"khớp cục bộ trên kiểu thêm nhiễu: Accuracy ≈ {acc0:.3f} {lam:+.2f} · D")

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 4.2))
for k, (ten, ds) in enumerate(troi.items()):
    d, acc = np.array(ds).T
    ax.plot(d, 100 * acc, "o-", color=MAU[k], lw=2, label=ten)
d_ve = np.linspace(0, d_nhieu.max(), 50)
ax.plot(d_ve, 100 * (acc0 + lam * d_ve), color="0.3", ls="--", lw=1, label="khớp cục bộ, ba điểm đầu của kiểu nhiễu")
ax.set_xlabel("chỉ số trôi: phân kỳ Jensen-Shannon so với tổng thể gốc (nat)")
ax.set_ylabel("độ chính xác (%), đo trên máy này")
ax.set_title("Cùng một chỉ số trôi, hai kết cục ngược nhau", fontsize=10)
ax.legend(loc="lower left", fontsize=8)
plt.tight_layout()
plt.show()

Chỉ số trôi phát hiện được cả hai dịch chuyển, nhưng không cho biết dịch chuyển nào có hại. Đường khớp
trên kiểu nhiễu dự báo sai hướng cho kiểu đổi tỉ lệ lớp. Đó là lý do sách nói một đường suy giảm cục bộ
chỉ có giá trị khi được khớp với kết quả thật, và cũng là cạm bẫy "coi mọi báo động trôi là lệnh quay lại
phiên bản trước": phản ứng phải theo nguyên nhân đã chẩn đoán, không theo riêng tín hiệu trôi.

### Nguyên lý 11: chẩn đoán lệch giữa huấn luyện và phục vụ

$S_{skew} = \mathbb{E}[d(f_{serve}(X), f_{train}(X))]$ so đầu ra của đường phục vụ với đầu ra của đường
huấn luyện trên cùng đầu vào, nên không cần nhãn. Chương 4 và chương 13 đều dành chỗ cho nó. Chương kết
mở đầu bằng đúng một ca như vậy: mỗi đội đạt chỉ số của mình, nhưng độ chính xác giảm 4 điểm phần trăm
trên vài nhóm thiết bị, vì sơ đồ lượng tử hoá tương tác với một đường tiền xử lý riêng của firmware. Ô
dưới dựng bốn đường tiền xử lý giả định của firmware và đo cả độ lệch lẫn độ chính xác.

In [ ]:
DUONG_FIRMWARE = {  # giả định của sổ tay
    "giống hệt lúc huấn luyện": lambda x: x,
    "làm tròn điểm ảnh về 4 mức": lambda x: np.round(x * 3) / 3,
    "tăng sáng 1.6 lần rồi cắt": lambda x: np.clip(x * 1.6, 0, 1),
    "nhị phân hoá ở 0.5": lambda x: (x > 0.5).astype(float),
    "lệch một điểm ảnh sang phải": lambda x: np.roll(x.reshape(-1, 8, 8), 1, axis=2).reshape(-1, 64),
}
p_train = mo_hinh.predict_proba(X_so[kt])
lech = {}
for ten, f in DUONG_FIRMWARE.items():
    p_serve = mo_hinh.predict_proba(f(X_so[kt]))
    khac = (p_serve.argmax(1) != p_train.argmax(1)).mean()     # d = 0/1 trên nhãn dự đoán
    acc = (p_serve.argmax(1) == y_so[kt]).mean()
    lech[ten] = (khac, acc)
    do_tren_may(f"{ten:>28}: S_skew (tỉ lệ dự đoán khác)", 100 * khac, "%")
    do_tren_may(f"{'':>28}  độ chính xác", 100 * acc, "%")

Độ lệch đo được mà không cần một nhãn nào, nên theo dõi được ngay trong sản xuất. Nhưng nó là một chỉ báo
rủi ro chứ không phải độ chính xác bị mất: mỗi dự đoán khác đi có thể đúng hoặc sai, nên cùng một mức
lệch có thể đi với độ chính xác giảm ít hay giảm nhiều. Muốn biết chất lượng thì cần nhãn.

### Nguyên lý 13: mô hình phản hồi thiên lệch

Bảng 2 viết $\Delta_g(k) \approx \Delta_g(0)\,\alpha_{fb}^k$ với $\alpha_{fb}$ khớp từ dữ liệu, và sách
đặt ba điều kiện: hệ số phản hồi đo được, gần như không đổi, và không có can thiệp hiệu quả nào. Chương 15
dựng các chỉ số tách theo nhóm mà nguyên lý này cần. Ô dưới mô phỏng một vòng phản hồi giả định: chênh
lệch giữa hai nhóm lớn dần qua mỗi vòng huấn luyện lại, rồi một can thiệp ở vòng 6 đổi hệ số. Mô hình khớp
trên sáu vòng đầu.

In [ ]:
r_fb = np.random.default_rng(SEED + 3)
VONG = np.arange(13)
ALPHA_TRUOC, ALPHA_SAU, VONG_CAN_THIEP, DELTA_0 = 1.18, 0.85, 6, 2.0   # giả định của sổ tay
delta = [DELTA_0]
for k in VONG[1:]:
    a_k = ALPHA_TRUOC if k <= VONG_CAN_THIEP else ALPHA_SAU
    delta.append(delta[-1] * a_k * float(np.exp(r_fb.normal(0, 0.04))))
delta = np.array(delta)
he_so, chan = np.polyfit(VONG[:VONG_CAN_THIEP], np.log(delta[:VONG_CAN_THIEP]), 1)
alpha_khop = float(np.exp(he_so))
du_bao = np.exp(chan) * alpha_khop ** VONG
print(f"α khớp trên vòng 0 tới {VONG_CAN_THIEP - 1}: {alpha_khop:.3f} (mô phỏng dùng {ALPHA_TRUOC})")
print(f"vòng 12: mô hình dự báo {du_bao[-1]:.1f} điểm, mô phỏng sau can thiệp cho {delta[-1]:.1f} điểm")

fig, ax = plt.subplots(figsize=(8.5, 4))
ax.plot(VONG, delta, "o-", color=MAU[1], lw=2, label="chênh lệch giữa hai nhóm (mô phỏng)")
ax.plot(VONG, du_bao, color="0.3", ls="--", lw=1.2, label=f"Δ(0) · α^k, α khớp = {alpha_khop:.2f}")
ax.axvline(VONG_CAN_THIEP + 0.5, color=XAM, lw=1)
ax.text(VONG_CAN_THIEP + 0.7, du_bao.max() * 0.97, "can thiệp", fontsize=8, color=XAM, va="top")
ax.set_xlabel("vòng huấn luyện lại k")
ax.set_ylabel("chênh lệch (điểm phần trăm, giả định)")
ax.set_title("Mô hình hàm mũ chỉ đúng khi hệ số không đổi và không ai can thiệp", fontsize=10)
ax.legend(loc="upper left", fontsize=8)
plt.tight_layout()
plt.show()

Mô hình khớp tốt trên đoạn nó được khớp, và sai ngay khi một trong ba điều kiện của sách không còn. Ở đây
điều kiện bị phá là điều ta muốn: một can thiệp hiệu quả. Theo dõi tách theo nhóm là thứ cho biết khi nào
điều đó xảy ra.

## 4. Phần I, nền tảng: dữ liệu là mã nguồn, trọng lực dữ liệu

### Nguyên lý 1: dữ liệu là mã nguồn

Bảng 2 viết hành vi là một hàm của dữ liệu, thuật toán, mã và tính ngẫu nhiên, rồi ghi: dữ liệu đổi hành
vi, và các đầu vào khác cũng có phần. Chương 1 đặt nó cạnh cách nhìn Software 2.0, và chương 4 dựng cả kỹ
thuật dữ liệu quanh nó. Ô dưới giữ nguyên mọi thứ trừ một đầu vào, rồi đo phần ảnh kiểm tra mà hai mô hình
dự đoán khác nhau.

In [ ]:
nua_1, nua_2 = hl[:500], hl[500:]
trung_binh, do_lech = X_so[nua_1].mean(0), X_so[nua_1].std(0) + 1e-6


def mlp(chi_so, hat_giong=SEED, chuan_hoa=False):
    X = (X_so[chi_so] - trung_binh) / do_lech if chuan_hoa else X_so[chi_so]
    return MLPClassifier((64,), max_iter=800, random_state=hat_giong).fit(X, y_so[chi_so])


goc_mh = mlp(nua_1)
p_goc = goc_mh.predict(X_so[kt])
bien_the = {
    "đổi dữ liệu (nửa kia của tập huấn luyện)": mlp(nua_2).predict(X_so[kt]),
    "đổi tính ngẫu nhiên (hạt giống khác)": mlp(nua_1, hat_giong=SEED + 1).predict(X_so[kt]),
    "đổi mã (chuẩn hoá đầu vào khác)": mlp(nua_1, chuan_hoa=True).predict((X_so[kt] - trung_binh) / do_lech),
}
do_tren_may("độ chính xác của mô hình gốc", 100 * (p_goc == y_so[kt]).mean(), "%")
for ten, p in bien_the.items():
    do_tren_may(f"{ten}: độ chính xác", 100 * (p == y_so[kt]).mean(), "%")
    do_tren_may(f"{'':>{len(ten)}}  tỉ lệ dự đoán khác mô hình gốc", 100 * (p != p_goc).mean(), "%")

Cả ba thay đổi đều đổi hành vi trên một phần ảnh. Tỉ lệ dự đoán khác nhau không bao giờ nhỏ hơn mức chênh
độ chính xác, và thường lớn hơn nhiều: hai mô hình có thể đúng gần cùng số ảnh mà không đúng cùng những ảnh
ấy. Đó là ý của cả hai vế trong bảng 2: dữ liệu là một đầu vào của hành vi ngang với mã, và nó không phải
đầu vào duy nhất.

### Nguyên lý 2: trọng lực dữ liệu

Đưa tính toán tới gần dữ liệu khi chi phí truyền dữ liệu lặp lại vượt chi phí đặt tính toán. Chương 2 dựng
nó từ bất biến về tính cục bộ của dữ liệu, chương 4 phát triển nó, và DLRM với bảng embedding cỡ terabyte
là ví dụ chương kết nêu. Chương không cho con số nào, nên mọi giá trị dưới đây là giả định của sổ tay.

In [ ]:
def nen_dua_tinh_toan_toi(v_byte, so_lan_quet, bw_mang, gio_dat_tinh_toan):
    """(giờ truyền dữ liệu lặp lại, giờ đặt tính toán cạnh dữ liệu, có nên dời tính toán không)."""
    gio_truyen = so_lan_quet * v_byte / bw_mang / 3600
    return gio_truyen, gio_dat_tinh_toan, gio_truyen > gio_dat_tinh_toan


V, BW_MANG, GIO_DAT = 20e12, 10e9 / 8, 24.0   # giả định: 20 TB, mạng 10 Gb/s, một ngày để dựng cụm cạnh dữ liệu
for lan in (1, 2, 5, 20):
    truyen, dat, nen = nen_dua_tinh_toan_toi(V, lan, BW_MANG, GIO_DAT)
    print(f"quét {lan:>2} lần: truyền {truyen:6.1f} giờ · đặt tính toán {dat:.0f} giờ → "
          f"{'dời tính toán tới dữ liệu' if nen else 'chuyển dữ liệu'}")
print(f"điểm hoà vốn: {GIO_DAT / (V / BW_MANG / 3600):.2f} lần quét")

Kết luận đổi theo mức tái dùng, đúng như cột "dự đoán gì" của bảng 2: tuỳ khối lượng, mức tái dùng, mạng
và khả năng di chuyển của tính toán. Đây là một phép so chi phí, không phải một luật đặt chỗ phổ quát.

## 5. Khung tích hợp: lần theo một quyết định lượng tử hoá

Sách viết mười ba nguyên lý không phải một danh sách để áp lần lượt, mà là một mạng lưới ràng buộc lẫn
nhau, và minh hoạ bằng một quyết định lượng tử hoá từ FP16 xuống INT8. Ô dưới lần theo quyết định ấy trên
ví dụ xuyên suốt, mỗi dòng một nguyên lý mà nó chạm tới. Dòng kiểm định thì không tính được ở đây: nó cần đo đường phục vụ
đã lượng tử hoá trên tổng thể triển khai, như nguyên lý 9 đòi.

Tiếp đó, theo phần C của bài thực hành đi kèm chương, ta giữ nguyên thiết kế và siết dần một giới hạn:
một ngân sách cho mỗi token, giả định 25 ms, nhân với một hệ số từ 1.05 xuống 0.55.

In [ ]:
def lan_theo_luong_tu_hoa(byte_truoc=BYTE_FP16, byte_sau=BYTE_INT8, batch=1):
    truoc = dinh_luat_sat(THAM_SO * byte_truoc, BW, batch * O, R_PEAK)
    sau = dinh_luat_sat(THAM_SO * byte_sau, BW, batch * O, R_PEAK)
    i_truoc, _, che_do_truoc = hop_dong_silicon(batch * O, THAM_SO * byte_truoc, R_PEAK, BW)
    i_sau, _, che_do_sau = hop_dong_silicon(batch * O, THAM_SO * byte_sau, R_PEAK, BW)
    e_truoc = nang_luong([THAM_SO / batch], [TI_LE_DRAM[0]])
    e_sau = nang_luong([THAM_SO * byte_sau / byte_truoc / batch], [TI_LE_DRAM[0]])
    return [
        ("5 biên Pareto", f"byte mỗi giá trị {byte_truoc:g} → {byte_sau:g}; đổi lại: sai số số học"),
        ("4 hợp đồng với silicon", f"mật độ tính toán {i_truoc:g} → {i_sau:g} FLOP/byte; " +
         (f"vẫn {che_do_sau}" if che_do_sau == che_do_truoc else f"{che_do_truoc} → {che_do_sau}")),
        ("3, 6 định luật sắt, mật độ", f"cận dưới mỗi bước {1e3 * truoc['chong_het']:.2f} → {1e3 * sau['chong_het']:.2f} ms"),
        ("7 năng lượng di chuyển", f"năng lượng DRAM mỗi token × {e_sau / e_truoc:.2f} (với cùng số lần truy cập mỗi byte)"),
        ("8 Amdahl", f"tăng tốc cả bước {truoc['noi_tiep'] / sau['noi_tiep']:.2f} lần, vì số hạng bộ nhớ chi phối"),
        ("12 ngân sách độ trễ", "đạt hay vượt tuỳ ngân sách đã chọn: xem phép siết bên dưới"),
        ("9 khoảng trống kiểm chứng", "không tính được ở đây: phải đo lại đường phục vụ INT8 trên tổng thể triển khai"),
    ], truoc, sau


dong, truoc, sau = lan_theo_luong_tu_hoa()
for nguyen_ly, ket_qua in dong:
    print(f"{nguyen_ly:>27}: {ket_qua}")

NGAN_SACH_TOKEN = 25e-3   # giả định: ngân sách cận dưới cho mỗi token, giây
he_so_siet = np.round(np.arange(1.05, 0.54, -0.05), 2)
print("\nsiết ngân sách mỗi token, giữ nguyên thiết kế (phần C của bài thực hành):")
for h in he_so_siet:
    gioi_han = NGAN_SACH_TOKEN * h
    print(f"  {h:.2f} × {1e3 * NGAN_SACH_TOKEN:.0f} ms = {1e3 * gioi_han:5.2f} ms: "
          f"FP16 {'đạt' if truoc['chong_het'] <= gioi_han else 'VƯỢT'} · INT8 {'đạt' if sau['chong_het'] <= gioi_han else 'VƯỢT'}")

Một quyết định chạm tới tám nguyên lý, và chỉ dòng cuối là không tính được ở đây. Dòng ấy là chỗ độ phức
tạp đổi chỗ: lượng tử hoá giảm lưu lượng bộ nhớ, và chuyển một phần gánh nặng sang kiểm định. Nguyên tắc bảo toàn độ phức tạp
của chương nhắc đúng điều đó, và sách nói rõ nó là một nguyên tắc chẩn đoán chứ không phải một định luật
bảo toàn: **không có gì để tính** ở chính nguyên tắc ấy, vì nó không hứa một đại lượng nào được giữ
nguyên. Thứ tính được là từng chi phí bị dời đi, như các dòng trên.

Phép siết cho thấy một khuyến nghị có biên giới. Qua mọi dòng, cận dưới của mỗi thiết kế không đổi, chỉ
phán quyết đổi: thiết kế FP16 đạt ngân sách cho tới một mức siết rồi vượt, còn INT8, với cận dưới bằng một
nửa, đạt trong cả khoảng siết mà bài thực hành cho phép. Biết biên giới ấy nằm ở đâu là biết khi nào phải
đánh giá lại.

## 6. Đổi phạm vi: bảng 1, cả cụm, và một hệ thống ghép

### Bảng 1: hành trình của MobileNetV2

Bảng 1 lần theo một mô hình qua bảy giai đoạn. Hai hàng của nó tính lại được: tích chập tách theo chiều
sâu giảm số FLOP của một tầng 3×3 có 256 kênh ra, và lượng tử hoá INT8, với dung lượng mỗi giá trị của
FP32 và FP16 so với INT8. Phép giảm của tích chập tách theo chiều sâu là $K^2 N / (K^2 + N)$, với $K$ là
cạnh nhân và $N$ là số kênh ra, như chương 6 đã dựng.

In [ ]:
K = sach(3, trich="3-by-3, 256-output-channel layer")
N_KENH = sach(256, trich="3-by-3, 256-output-channel layer")
theo_sach("FLOP giảm nhờ tích chập tách theo chiều sâu (lần)", K**2 * N_KENH / (K**2 + N_KENH), sach=8.7,
          nguon="8.7× · 13.7×")
theo_sach("FP32 so với INT8, dung lượng mỗi giá trị", SO_BIT["FP32"] / SO_BIT["INT8"], sach=4, nguon="4× · 2×")
theo_sach("FP16 so với INT8, dung lượng mỗi giá trị", SO_BIT["FP16"] / SO_BIT["INT8"], sach=2, nguon="4× · 2×")

Con số thứ hai của hàng tích chập, 13.7 lần ít phép toán hơn ResNet-50 ở quy mô ImageNet, cần số phép toán
của cả hai mô hình mà chương không in, nên sổ tay không tính lại nó.

### Từ một node tới cả cụm

Chương kết khép lại bằng một ghi chú về quy mô. Theo mô hình hỏng độc lập, giống hệt nhau, với tỉ lệ hỏng
không đổi, và coi lần hỏng GPU đầu tiên là một sự kiện của cả cụm, thời gian trung bình giữa hai lần hỏng
của cụm bằng thời gian trung bình tới khi hỏng của một GPU chia cho số GPU. Sách dùng MTTF tham chiếu 5.7
năm cho một GPU và một cụm 1,024 GPU.

In [ ]:
MTTF_NAM = sach(5.7, nguon="5.7 năm · 1,024 GPU · khoảng 48.8 giờ")
SO_GPU_CUM = sach(1024, nguon="5.7 năm · 1,024 GPU · khoảng 48.8 giờ")
GIO_MOI_NAM = 365 * 24   # hằng số lịch
mtbf = MTTF_NAM * GIO_MOI_NAM / SO_GPU_CUM
theo_sach("MTBF của cụm (giờ)", mtbf, sach=48.8, nguon="5.7 năm · 1,024 GPU · khoảng 48.8 giờ")
for tuan in (1, 2, 4):   # giả định: độ dài lần chạy
    print(f"một lần chạy {tuan} tuần: xác suất gặp ít nhất một lần hỏng {1 - math.exp(-tuan * 7 * 24 / mtbf):.2%}")
EXA = sach(1e18, nguon="từ 10^18 FLOP/s")
print(f"10^18 FLOP/s duy trì cần ít nhất {EXA / (R_PEAK / SO_GPU):,.0f} H100 chạy ở thông lượng đỉnh FP16, "
      f"trước khi tính hiệu suất phần cứng")

Một lần chạy nhiều tuần gần như chắc chắn gặp hỏng hóc, nên chịu lỗi là một yêu cầu vận hành. Vật lý không
đổi, chỉ có quy mô mà nó bắt đầu ràng buộc là đổi.

### Một hệ thống ghép nhiều thành phần

Một yêu cầu đi qua truy xuất, lập kế hoạch, hai công cụ, sinh câu trả lời và kiểm chứng. Sách viết độ trễ
thực tế đi theo đường tới hạn của yêu cầu, với hai công cụ chạy song song chỉ góp phần lớn hơn chứ không
phải tổng, cộng thêm phụ trội điều phối. Chương không cho con số nào ngoài câu tự kiểm tra, nên mọi độ trễ
và độ tin cậy dưới đây là giả định của sổ tay.

In [ ]:
CHANG = [("truy xuất", 40.0), ("lập kế hoạch", 90.0), (("công cụ 1", 60.0), ("công cụ 2", 150.0)),
         ("sinh câu trả lời", 200.0), ("kiểm chứng", 25.0)]   # ms, giả định; bộ đôi là chạy song song
PHU_TROI_DIEU_PHOI = 15.0   # ms, giả định


def duong_toi_han(chang, phu_troi=0.0):
    """Độ trễ theo đường tới hạn: chặng song song góp phần lớn nhất, chặng nối tiếp cộng dồn."""
    return sum(max(t for _, t in c) if isinstance(c[0], tuple) else c[1] for c in chang) + phu_troi


def tong_noi_tiep(chang, phu_troi=0.0):
    return sum(sum(t for _, t in c) if isinstance(c[0], tuple) else c[1] for c in chang) + phu_troi


so_thanh_phan = sum(len(c) if isinstance(c[0], tuple) else 1 for c in CHANG)
print(f"đường tới hạn: {duong_toi_han(CHANG, PHU_TROI_DIEU_PHOI):.0f} ms · nếu chạy hai công cụ nối tiếp: "
      f"{tong_noi_tiep(CHANG, PHU_TROI_DIEU_PHOI):.0f} ms")
for p in (0.999, 0.995, 0.99):   # giả định: mỗi thành phần thành công với xác suất p, độc lập
    print(f"{so_thanh_phan} thành phần, mỗi thành phần đúng {p:.1%}: cả yêu cầu đúng {p ** so_thanh_phan:.2%}")

Ghép thêm thành phần có thể tăng năng lực, nhưng mỗi giao diện thêm một chặng vào đường tới hạn và một
cách hỏng vào phép nhân độ tin cậy. Đó là lý do sách đòi mỗi giao diện có hành vi đo được.

### Những chỗ không có gì để tính

* **Bảo toàn độ phức tạp**: không có gì để tính, vì sách nói rõ đó là một nguyên tắc chẩn đoán chứ không
  phải một định luật bảo toàn, nên không có đại lượng nào để giữ nguyên hay đo. Phần 5 tính những chi phí
  bị dời đi thay cho nó.
* **Nền tảng di động tham chiếu nhỏ hơn khoảng 10 lần về bộ nhớ và 140 lần về công suất so với một bộ tăng
  tốc lớp H100**: không có gì để tính, vì chương chỉ cho tỉ số, không cho hai giá trị tuyệt đối.
* **AI vì lợi ích xã hội và trách nhiệm của người kỹ sư**: không có gì để tính, vì đây là những lập luận
  về miền ứng dụng và về ai chịu chi phí, mà chương nói phải do hiểu biết của từng miền quyết định.

## Thử thêm

1. Ở phần 2, đổi `SO_BIT["INT8"]` thành 4 (bốn bit mỗi tham số). Điểm gãy được chạm ở batch nào, và thời
   gian mỗi token ở batch 1 còn bao nhiêu? Các dòng ✗ của phép so với bảng 1 khi ấy là điều được chờ đợi.
2. Ở nguyên lý 7, đổi giả định một lần truy cập đọc một giá trị FP16 thành một lần đọc 64 byte (một dòng
   bộ nhớ đệm): chia `THAM_SO / b` cho 32. Batch nào thì DRAM thôi chi phối?
3. Ở nguyên lý 10, đổi `lop_de` thành ba lớp khó nhất (`np.argsort(theo_lop)[:3]`). Kiểu đổi tỉ lệ lớp
   còn làm độ chính xác tăng không?
4. Ở phần 6, đổi `SO_GPU_CUM` thành 16384. MTBF của cụm còn bao nhiêu giờ? Dòng ✗ khi ấy là điều được
   chờ đợi.

## Tóm lại

* Một token của Llama 2 70B ở batch 1 cần ít nhất 20.9 ms để đọc trọng số và 0.07 ms để tính: tỉ số 295.2
  lần ấy chính là điểm gãy chia cho mật độ tính toán, nên định luật sắt và hợp đồng với silicon là cùng một
  phép so.
* Gom batch tăng mật độ tính toán cho tới điểm gãy, lượng tử hoá giảm byte; năng lượng theo đúng số lần
  truy cập, và Amdahl cho biết tối ưu số hạng nhỏ thì không được gì.
* Biên Pareto không chọn hộ một điểm. Ngân sách độ trễ ở một phân vị đã chọn mới chọn, và một trung bình
  50 ms có thể đi cùng một phân vị 99 gấp 40 lần.
* Các nguyên lý triển khai là chẩn đoán: kiểm chứng chỉ đúng trên tổng thể đã nêu, chỉ số trôi không cho
  biết chiều của chất lượng, độ lệch là rủi ro chứ không phải mất mát, và mô hình phản hồi hàm mũ chỉ đúng
  khi không ai can thiệp.
* Khi phạm vi đổi, các phép tính vẫn vậy: 5.7 năm cho một GPU thành khoảng 48.8 giờ cho 1,024 GPU, và một
  hệ thống ghép đi theo đường tới hạn của nó.

---
### Nguồn và giấy phép

Sổ tay này đi kèm bài học [Kết luận — lập luận xuyên ranh giới](https://d3lu8vk4we0dc.cloudfront.net/lessons/vol1-ch16-conclusion/), dựng từ chương
[*Conclusion*](https://mlsysbook.ai/vol1/conclusion/conclusion.html) của sách. Một phần sổ tay phỏng theo phòng thí nghiệm `labs/vol1/lab_16_ml_conclusion.py` của sách (© 2026 President and Fellows of Harvard College, CC BY-NC-SA 4.0).

Nội dung gốc thuộc Machine Learning Systems của Vijay Janapa Reddi và cộng sự, MIT Press. Bản quyền © 2024-2026 Harvard University. Giấy phép CC BY-NC-SA 4.0. Bản tiếng Việt này là tác phẩm phái sinh dùng cùng giấy phép: ghi công, chia sẻ tương tự, phi thương mại.